# Bayesian Regime Allocation

Uses Bayesian regime estimates to adapt allocation in a historical backtest.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
================================================================================
REGIME-ADAPTIVE TRADING SYSTEM WITH BCPM V2.0 - PRODUCTION VERSION
================================================================================

Complete integrated system:
1. Regime detection using BCPM on SPY ONLY (no other assets analyzed)
2. Three regime-specific portfolio strategies on expanded universe:
   - BULLISH (Low Vol <1%): OPS Momentum @ 120% exposure
   - NEUTRAL (Med Vol 1-2%): Hierarchical Risk Parity @ 100% exposure
   - BEARISH (High Vol >2%): Risk Parity + SH Short @ 55% exposure (40% long + 15% short)

Author: Trading Strategies Module
Version: 2.0 Final - SPY-Only Regime Detection
Date: October 2025
================================================================================
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from scipy import stats
from scipy.special import logsumexp
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf
from typing import List, Tuple, Dict, Optional
import json
from collections import Counter

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")


class Config:
    """
    Master configuration for regime-adaptive trading system.
    Modify these parameters to customize the system.
    """

    REGIME_TICKER = "SPY"
    VIX_TICKER = "^VIX"
    TRADING_TICKERS = [
        "SPY",
        "QQQ",
        "DIA",
        "IWM",
        "VTI",
        "XLK",
        "XLF",
        "XLE",
        "XLV",
        "XLY",
        "XLP",
        "XLI",
        "XLB",
        "XLU",
        "EFA",
        "EEM",
        "VEA",
        "VWO",
        "EWJ",
        "FXI",
        "TLT",
        "IEF",
        "SHY",
        "LQD",
        "HYG",
        "TIP",
        "AGG",
        "GLD",
        "SLV",
        "DBC",
        "USO",
        "VNQ",
        "REET",
    ]
    SHORT_TICKER = "SH"
    START_DATE = "2013-12-31"
    END_DATE = "2018-12-31"
    RV_WINDOW = 21
    RV_SHORT_WINDOW = 5
    PARKINSON_WINDOW = 21
    VOLUME_MA_WINDOW = 21
    VIX_SLOPE_WINDOW = 10
    HAZARD_RATE_FAST = 1 / 63
    HAZARD_RATE_MEDIUM = 1 / 125
    HAZARD_RATE_SLOW = 1 / 252
    USE_ADAPTIVE_HAZARD = True
    HAZARD_VIX_WEIGHT = 0.3
    HAZARD_VOLUME_WEIGHT = 0.2
    HAZARD_BASE_RATE = 1 / 125
    PRIOR_MU = 0.0
    PRIOR_KAPPA = 1.0
    PRIOR_ALPHA = 3.0
    PRIOR_BETA = 0.002
    LOW_VOL_THRESHOLD = 0.01
    HIGH_VOL_THRESHOLD = 0.02
    REGIME_SMOOTHING_WINDOW = 3
    ENSEMBLE_VOTING_METHOD = "weighted"
    ENSEMBLE_FAST_WEIGHT = 0.4
    ENSEMBLE_MEDIUM_WEIGHT = 0.4
    ENSEMBLE_SLOW_WEIGHT = 0.2
    CP_THRESHOLD_HIGH = 0.3
    CP_THRESHOLD_MEDIUM = 0.2
    MIN_HISTORY = 63
    REBALANCE_FREQUENCY = 21
    BULLISH_EXPOSURE = 1.2
    MOMENTUM_LOOKBACK = [21, 63, 126, 252]
    MOMENTUM_WEIGHTS = [0.4, 0.3, 0.2, 0.1]
    TOP_N_MOMENTUM = 8
    MIN_MOMENTUM_SCORE = 0.0
    NEUTRAL_EXPOSURE = 1.0
    HRP_LINKAGE_METHOD = "ward"
    BEARISH_LONG_EXPOSURE = 0.4
    BEARISH_SHORT_EXPOSURE = 0.15
    BEARISH_TOTAL_EXPOSURE = BEARISH_LONG_EXPOSURE + BEARISH_SHORT_EXPOSURE
    VOLATILITY_TARGET = 0.12
    MAX_WEIGHT_SINGLE_ASSET = 0.25
    MIN_WEIGHT_SINGLE_ASSET = 0.01
    COMMISSION_PCT = 0.001
    SAVE_RESULTS = True
    OUTPUT_DIR = "regime_adaptive_backtest/"
    PLOT_SPY_REGIME = True


class YahooDataDownloader:
    """
    Downloads historical price data from Yahoo Finance.
    Handles batch downloads and error checking.
    """

    def __init__(self):
        """
        Initialize downloader with API key.

        Parameters:
        -----------
            Yahoo Finance key for authentication
        """

    def download_ticker(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Download historical daily data for a single ticker.

        Parameters:
        -----------
        ticker : str
            Ticker symbol (e.g., 'SPY', '^VIX')
        start_date : str
            Start date in 'YYYY-MM-DD' format
        end_date : str
            End date in 'YYYY-MM-DD' format

        Returns:
        --------
        pd.DataFrame
            DataFrame with OHLCV data and adjusted close
        """
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) == 0:
                print(f"  WARNING: No data for {ticker}")
                return pd.DataFrame()
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df["ticker"] = ticker
            return df
        except Exception as e:
            print(f"  ERROR downloading {ticker}: {str(e)}")
            return pd.DataFrame()

    def download_multiple_tickers(
        self, tickers: List[str], start_date: str, end_date: str
    ) -> Dict[str, pd.DataFrame]:
        """
        Download data for multiple tickers with progress tracking.

        Parameters:
        -----------
        tickers : List[str]
            List of ticker symbols
        start_date : str
            Start date
        end_date : str
            End date

        Returns:
        --------
        Dict[str, pd.DataFrame]
            Dictionary mapping ticker to DataFrame
        """
        data_dict = {}
        total = len(tickers)
        print(f"Downloading {total} tickers...")
        for i, ticker in enumerate(tickers, 1):
            print(f"  [{i}/{total}] {ticker}...", end="")
            df = self.download_ticker(ticker, start_date, end_date)
            if not df.empty:
                data_dict[ticker] = df
                print(f" ✓ ({len(df)} records)")
            else:
                print(f" ✗ (no data)")
        success_rate = 100 * len(data_dict) / total
        print(f"\nDownloaded {len(data_dict)}/{total} tickers ({success_rate:.1f}% success)")
        return data_dict


class EnhancedFeatureEngine:
    """
    Enhanced feature engineering with volatility estimators.
    All features properly lagged to avoid look-ahead bias.
    """

    @staticmethod
    def compute_returns(df: pd.DataFrame) -> pd.DataFrame:
        """
        Compute log returns from adjusted close prices.

        Returns at time T use Close[T] and Close[T-1].
        """
        df = df.copy()
        df["ret_close"] = np.log(df["adjClose"] / df["adjClose"].shift(1))
        return df

    @staticmethod
    def compute_realized_volatility(df: pd.DataFrame, window: int = 21) -> pd.DataFrame:
        """
        Compute realized volatility using close-to-close returns.
        Annualized using sqrt(252) convention.
        """
        df = df.copy()
        df[f"rv_{window}"] = df["ret_close"].rolling(window=window).std() * np.sqrt(252)
        return df

    @staticmethod
    def compute_parkinson_volatility(df: pd.DataFrame, window: int = 21) -> pd.DataFrame:
        """
        Compute Parkinson volatility estimator using high-low range.
        More efficient than close-to-close for capturing intraday volatility.
        """
        df = df.copy()
        df["log_hl"] = np.log(df["high"] / df["low"])
        const = 1 / (4 * np.log(2))
        df[f"parkinson_{window}"] = np.sqrt(
            const * df["log_hl"].pow(2).rolling(window=window).sum() / window
        ) * np.sqrt(252)
        return df

    @staticmethod
    def compute_volume_ratio(df: pd.DataFrame, window: int = 21) -> pd.DataFrame:
        """
        Compute volume ratio and volume surge indicator.
        High volume during selloffs indicates panic/forced selling.
        """
        df = df.copy()
        df["volume_ma"] = df["volume"].rolling(window=window).mean()
        df["volume_ratio"] = df["volume"] / df["volume_ma"]
        df["volume_surge"] = (df["volume_ratio"] > 2.0).astype(int)
        return df

    @staticmethod
    def compute_vix_features(vix_df: pd.DataFrame, window: int = 10) -> pd.DataFrame:
        """
        Compute VIX-based features including term structure slope.
        VIX slope: positive = backwardation (fear), negative = contango (calm).
        """
        df = vix_df.copy()
        df["vix_ma"] = df["adjClose"].rolling(window=window).mean()
        df["vix_ratio"] = df["adjClose"] / df["vix_ma"]
        df["vix_slope"] = df["adjClose"].diff(window) / window
        df["vix_elevated"] = (df["adjClose"] > 20).astype(int)
        df["vix_spike"] = (df["adjClose"] > 30).astype(int)
        return df

    @staticmethod
    def lag_features(df: pd.DataFrame, lag: int = 1) -> pd.DataFrame:
        """
        Apply lag to ALL features to prevent look-ahead bias.

        CRITICAL: Features at T computed from data up to T-1, must shift by 1 day.
        Decision at open of day T uses features from close of day T-1.
        """
        df = df.copy()
        exclude_cols = ["date", "adjClose", "open", "high", "low", "close", "volume", "ticker"]
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        for col in feature_cols:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
        df = df.drop(columns=feature_cols)
        return df


class AdaptiveBayesianOnlineChangepoint:
    """
    Adaptive BOCD with dynamic hazard function based on market stress.

    Key enhancement: hazard rate adapts to VIX and volume conditions.
    When VIX spikes or volume surges, model becomes more sensitive to changepoints.
    """

    def __init__(
        self,
        base_hazard_rate: float = 1 / 125,
        adaptive: bool = True,
        vix_weight: float = 0.3,
        volume_weight: float = 0.2,
        prior_mu: float = 0.0,
        prior_kappa: float = 1.0,
        prior_alpha: float = 3.0,
        prior_beta: float = 0.002,
    ):
        """Initialize adaptive BOCD with prior parameters."""
        self.base_hazard_rate = base_hazard_rate
        self.adaptive = adaptive
        self.vix_weight = vix_weight
        self.volume_weight = volume_weight
        self.prior_mu = prior_mu
        self.prior_kappa = prior_kappa
        self.prior_alpha = prior_alpha
        self.prior_beta = prior_beta
        self.run_length_log_probs = None
        self.sufficient_stats = None

    def compute_adaptive_hazard(self, vix_level: float = None, volume_ratio: float = None) -> float:
        """
        Compute adaptive hazard rate based on market conditions.

        Logic:
        - High VIX → increase hazard (more likely to have regime change)
        - High volume → increase hazard (potential forced selling/panic)
        """
        if not self.adaptive:
            return self.base_hazard_rate
        hazard_multiplier = 1.0
        if vix_level is not None and (not np.isnan(vix_level)):
            vix_factor = min((vix_level - 10) / 30, 1.0)
            vix_factor = max(vix_factor, 0.0)
            hazard_multiplier += self.vix_weight * vix_factor
        if volume_ratio is not None and (not np.isnan(volume_ratio)):
            vol_factor = min((volume_ratio - 1.0) / 2.0, 1.0)
            vol_factor = max(vol_factor, 0.0)
            hazard_multiplier += self.volume_weight * vol_factor
        adaptive_hazard = self.base_hazard_rate * hazard_multiplier
        return min(adaptive_hazard, 0.5)

    def initialize(self, max_run_length: int = 1000):
        """Initialize run length distribution and sufficient statistics."""
        self.run_length_log_probs = np.log(np.zeros(max_run_length + 1))
        self.run_length_log_probs[0] = 0
        self.sufficient_stats = {
            "n": np.zeros(max_run_length + 1),
            "mean": np.full(max_run_length + 1, self.prior_mu),
            "sum_sq_dev": np.zeros(max_run_length + 1),
        }
        self.max_run_length = max_run_length

    def update(
        self, observation: float, vix_level: float = None, volume_ratio: float = None
    ) -> Tuple[float, np.ndarray, float]:
        """
        Update BOCD with new observation.

        Returns:
        --------
        Tuple[float, np.ndarray, float]
            - Changepoint probability
            - Run length distribution
            - Adaptive hazard rate used
        """
        if self.run_length_log_probs is None:
            raise ValueError("Must call initialize() before update()")
        hazard_rate = self.compute_adaptive_hazard(vix_level, volume_ratio)
        if np.isnan(observation):
            return (0.0, np.exp(self.run_length_log_probs), hazard_rate)
        log_pred_probs = np.zeros(self.max_run_length + 1)
        for r in range(len(self.run_length_log_probs)):
            if self.run_length_log_probs[r] > -np.inf:
                n = self.sufficient_stats["n"][r]
                mean = self.sufficient_stats["mean"][r]
                sum_sq_dev = self.sufficient_stats["sum_sq_dev"][r]
                post_kappa = self.prior_kappa + n
                post_alpha = self.prior_alpha + n / 2
                post_beta = (
                    self.prior_beta
                    + sum_sq_dev / 2
                    + self.prior_kappa * n * (mean - self.prior_mu) ** 2 / (2 * post_kappa)
                )
                df = 2 * post_alpha
                loc = mean
                scale = np.sqrt(post_beta * (post_kappa + 1) / (post_alpha * post_kappa))
                try:
                    log_pred_probs[r] = stats.t.logpdf(observation, df=df, loc=loc, scale=scale)
                except:
                    log_pred_probs[r] = -np.inf
        log_growth_probs = self.run_length_log_probs + log_pred_probs + np.log(1 - hazard_rate)
        log_cp_prob = logsumexp(self.run_length_log_probs + log_pred_probs + np.log(hazard_rate))
        new_run_length_log_probs = np.full(self.max_run_length + 1, -np.inf)
        new_run_length_log_probs[0] = log_cp_prob
        new_run_length_log_probs[1:] = log_growth_probs[:-1]
        new_run_length_log_probs -= logsumexp(new_run_length_log_probs)
        cp_prob = np.exp(new_run_length_log_probs[0])
        new_sufficient_stats = {
            "n": np.zeros(self.max_run_length + 1),
            "mean": np.zeros(self.max_run_length + 1),
            "sum_sq_dev": np.zeros(self.max_run_length + 1),
        }
        new_sufficient_stats["n"][0] = 0
        new_sufficient_stats["mean"][0] = self.prior_mu
        new_sufficient_stats["sum_sq_dev"][0] = 0
        for r in range(1, self.max_run_length + 1):
            if new_run_length_log_probs[r] > -np.inf:
                old_r = r - 1
                old_n = self.sufficient_stats["n"][old_r]
                old_mean = self.sufficient_stats["mean"][old_r]
                old_sum_sq_dev = self.sufficient_stats["sum_sq_dev"][old_r]
                new_n = old_n + 1
                new_mean = (old_n * old_mean + observation) / new_n
                new_sum_sq_dev = old_sum_sq_dev + old_n * (observation - old_mean) ** 2 / new_n
                new_sufficient_stats["n"][r] = new_n
                new_sufficient_stats["mean"][r] = new_mean
                new_sufficient_stats["sum_sq_dev"][r] = new_sum_sq_dev
        self.run_length_log_probs = new_run_length_log_probs
        self.sufficient_stats = new_sufficient_stats
        return (cp_prob, np.exp(self.run_length_log_probs), hazard_rate)

    def get_expected_run_length(self) -> float:
        """Get expected run length (time since last changepoint)."""
        run_lengths = np.arange(len(self.run_length_log_probs))
        probs = np.exp(self.run_length_log_probs)
        return np.sum(run_lengths * probs)

    def get_regime_volatility(self) -> float:
        """Get current regime's estimated volatility from sufficient statistics."""
        most_probable_r = np.argmax(self.run_length_log_probs)
        n = self.sufficient_stats["n"][most_probable_r]
        if n == 0:
            return np.nan
        sum_sq_dev = self.sufficient_stats["sum_sq_dev"][most_probable_r]
        variance = sum_sq_dev / n if n > 0 else 0
        return np.sqrt(variance)


class MultiScaleRegimeDetector:
    """
    Multi-scale BOCD with ensemble voting.
    Runs 3 parallel detectors at different time scales and aggregates.
    """

    @staticmethod
    def run_multiscale_bocd(
        data: np.ndarray,
        vix_series: np.ndarray = None,
        volume_series: np.ndarray = None,
        config: Config = None,
    ) -> Dict[str, np.ndarray]:
        """
        Run multi-scale BOCD on time series.

        Returns dict with cp_prob, run_length, regime_vol for each scale,
        plus ensemble aggregates.
        """
        if config is None:
            config = Config()
        bocd_fast = AdaptiveBayesianOnlineChangepoint(
            base_hazard_rate=config.HAZARD_RATE_FAST,
            adaptive=config.USE_ADAPTIVE_HAZARD,
            vix_weight=config.HAZARD_VIX_WEIGHT,
            volume_weight=config.HAZARD_VOLUME_WEIGHT,
            prior_mu=config.PRIOR_MU,
            prior_kappa=config.PRIOR_KAPPA,
            prior_alpha=config.PRIOR_ALPHA,
            prior_beta=config.PRIOR_BETA,
        )
        bocd_medium = AdaptiveBayesianOnlineChangepoint(
            base_hazard_rate=config.HAZARD_RATE_MEDIUM,
            adaptive=config.USE_ADAPTIVE_HAZARD,
            vix_weight=config.HAZARD_VIX_WEIGHT,
            volume_weight=config.HAZARD_VOLUME_WEIGHT,
            prior_mu=config.PRIOR_MU,
            prior_kappa=config.PRIOR_KAPPA,
            prior_alpha=config.PRIOR_ALPHA,
            prior_beta=config.PRIOR_BETA,
        )
        bocd_slow = AdaptiveBayesianOnlineChangepoint(
            base_hazard_rate=config.HAZARD_RATE_SLOW,
            adaptive=config.USE_ADAPTIVE_HAZARD,
            vix_weight=config.HAZARD_VIX_WEIGHT,
            volume_weight=config.HAZARD_VOLUME_WEIGHT,
            prior_mu=config.PRIOR_MU,
            prior_kappa=config.PRIOR_KAPPA,
            prior_alpha=config.PRIOR_ALPHA,
            prior_beta=config.PRIOR_BETA,
        )
        max_rl = min(1000, len(data))
        bocd_fast.initialize(max_run_length=max_rl)
        bocd_medium.initialize(max_run_length=max_rl)
        bocd_slow.initialize(max_run_length=max_rl)
        results = {
            "cp_prob_fast": [],
            "cp_prob_medium": [],
            "cp_prob_slow": [],
            "run_length_fast": [],
            "run_length_medium": [],
            "run_length_slow": [],
            "regime_vol_fast": [],
            "regime_vol_medium": [],
            "regime_vol_slow": [],
            "adaptive_hazards": [],
        }
        for i, obs in enumerate(data):
            vix_val = vix_series[i] if vix_series is not None else None
            vol_val = volume_series[i] if volume_series is not None else None
            cp_fast, _, hz_fast = bocd_fast.update(obs, vix_val, vol_val)
            cp_medium, _, hz_medium = bocd_medium.update(obs, vix_val, vol_val)
            cp_slow, _, hz_slow = bocd_slow.update(obs, vix_val, vol_val)
            results["cp_prob_fast"].append(cp_fast)
            results["cp_prob_medium"].append(cp_medium)
            results["cp_prob_slow"].append(cp_slow)
            results["run_length_fast"].append(bocd_fast.get_expected_run_length())
            results["run_length_medium"].append(bocd_medium.get_expected_run_length())
            results["run_length_slow"].append(bocd_slow.get_expected_run_length())
            results["regime_vol_fast"].append(bocd_fast.get_regime_volatility())
            results["regime_vol_medium"].append(bocd_medium.get_regime_volatility())
            results["regime_vol_slow"].append(bocd_slow.get_regime_volatility())
            results["adaptive_hazards"].append(np.mean([hz_fast, hz_medium, hz_slow]))
        for key in results:
            results[key] = np.array(results[key])
        if config.ENSEMBLE_VOTING_METHOD == "weighted":
            results["cp_prob_ensemble"] = (
                config.ENSEMBLE_FAST_WEIGHT * results["cp_prob_fast"]
                + config.ENSEMBLE_MEDIUM_WEIGHT * results["cp_prob_medium"]
                + config.ENSEMBLE_SLOW_WEIGHT * results["cp_prob_slow"]
            )
        else:
            votes = (
                (results["cp_prob_fast"] > config.CP_THRESHOLD_HIGH).astype(int)
                + (results["cp_prob_medium"] > config.CP_THRESHOLD_HIGH).astype(int)
                + (results["cp_prob_slow"] > config.CP_THRESHOLD_HIGH).astype(int)
            )
            results["cp_prob_ensemble"] = votes / 3.0
        results["regime_vol_ensemble"] = (
            config.ENSEMBLE_FAST_WEIGHT * results["regime_vol_fast"]
            + config.ENSEMBLE_MEDIUM_WEIGHT * results["regime_vol_medium"]
            + config.ENSEMBLE_SLOW_WEIGHT * results["regime_vol_slow"]
        )
        return results

    @staticmethod
    def classify_regimes_3level(
        regime_vols: np.ndarray,
        low_threshold: float = 0.01,
        high_threshold: float = 0.02,
        smoothing_window: int = 3,
    ) -> np.ndarray:
        """
        Classify into 3 regimes: 0=Low/Bullish, 1=Medium/Neutral, 2=High/Bearish.
        Apply smoothing to reduce noise in regime transitions.
        """
        regime_labels = np.zeros(len(regime_vols))
        regime_labels[regime_vols < low_threshold] = 0
        regime_labels[(regime_vols >= low_threshold) & (regime_vols <= high_threshold)] = 1
        regime_labels[regime_vols > high_threshold] = 2
        if smoothing_window > 1:
            regime_labels_smooth = (
                pd.Series(regime_labels)
                .rolling(window=smoothing_window, min_periods=1)
                .apply(
                    lambda x: Counter(x).most_common(1)[0][0] if len(x) > 0 else x.iloc[-1],
                    raw=False,
                )
                .values
            )
            return regime_labels_smooth.astype(int)
        return regime_labels.astype(int)


class OPSMomentumStrategy:
    """
    OPS (Optimization-based Portfolio Selection) Momentum Strategy.
    Multi-period momentum scoring with volatility scaling.
    Used in BULLISH (low volatility) regimes.
    """

    @staticmethod
    def compute_momentum_scores(
        returns_df: pd.DataFrame,
        lookbacks: List[int] = [21, 63, 126, 252],
        weights: List[float] = [0.4, 0.3, 0.2, 0.1],
    ) -> pd.Series:
        """
        Compute composite momentum score using weighted multi-period returns.
        Recent periods get higher weight.
        """
        scores = pd.Series(0.0, index=returns_df.columns)
        for lookback, weight in zip(lookbacks, weights):
            if len(returns_df) >= lookback:
                period_returns = returns_df.iloc[-lookback:].sum()
                scores += weight * period_returns
        return scores

    @staticmethod
    def construct_momentum_portfolio(
        returns_df: pd.DataFrame,
        top_n: int = 8,
        min_score: float = 0.0,
        max_weight: float = 0.25,
        target_vol: float = 0.12,
        current_vol: float = None,
    ) -> pd.Series:
        """
        Construct momentum-weighted portfolio.

        Steps:
        1. Compute momentum scores
        2. Select top N assets with positive momentum
        3. Weight proportional to momentum score
        4. Scale by volatility target
        """
        scores = OPSMomentumStrategy.compute_momentum_scores(returns_df)
        eligible = scores[scores > min_score].sort_values(ascending=False)
        if len(eligible) == 0:
            weights = pd.Series(1.0 / len(scores), index=scores.index)
            return weights
        selected = eligible.head(top_n)
        weights = selected / selected.sum()
        weights = weights.clip(upper=max_weight)
        weights = weights / weights.sum()
        full_weights = pd.Series(0.0, index=scores.index)
        full_weights[weights.index] = weights
        if current_vol is not None and current_vol > 0:
            vol_scalar = target_vol / current_vol
            vol_scalar = np.clip(vol_scalar, 0.5, 2.0)
        else:
            vol_scalar = 1.0
        return full_weights * vol_scalar


class HRPStrategy:
    """
    Hierarchical Risk Parity (HRP) Strategy.
    Uses hierarchical clustering for diversified portfolio construction.
    Used in NEUTRAL (medium volatility) regimes.
    """

    @staticmethod
    def compute_distance_matrix(corr_matrix: pd.DataFrame) -> np.ndarray:
        """Convert correlation matrix to distance matrix."""
        return np.sqrt((1 - corr_matrix) / 2)

    @staticmethod
    def get_quasi_diag(link: np.ndarray) -> List:
        """Get quasi-diagonal ordering from linkage matrix."""
        link = link.astype(int)
        sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
        num_items = link[-1, 3]
        while sort_ix.max() >= num_items:
            sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
            df0 = sort_ix[sort_ix >= num_items]
            i = df0.index
            j = df0.values - num_items
            sort_ix[i] = link[j, 0]
            df0 = pd.Series(link[j, 1], index=i + 1)
            sort_ix = pd.concat([sort_ix, df0])
            sort_ix = sort_ix.sort_index()
            sort_ix.index = range(sort_ix.shape[0])
        return sort_ix.tolist()

    @staticmethod
    def get_cluster_var(cov: pd.DataFrame, c_items: List) -> float:
        """Compute variance of cluster using inverse-variance weighting."""
        cov_slice = cov.loc[c_items, c_items]
        w = 1.0 / np.diag(cov_slice)
        w /= w.sum()
        return np.dot(w, np.dot(cov_slice, w))

    @staticmethod
    def get_rec_bipart(cov: pd.DataFrame, sort_ix: List) -> pd.Series:
        """Recursive bisection to allocate weights in HRP."""
        w = pd.Series(1.0, index=sort_ix)
        c_items = [sort_ix]
        while len(c_items) > 0:
            c_items = [
                i[j:k]
                for i in c_items
                for j, k in ((0, len(i) // 2), (len(i) // 2, len(i)))
                if len(i) > 1
            ]
            for i in range(0, len(c_items), 2):
                c_items0 = c_items[i]
                c_items1 = c_items[i + 1]
                c_var0 = HRPStrategy.get_cluster_var(cov, c_items0)
                c_var1 = HRPStrategy.get_cluster_var(cov, c_items1)
                alpha = 1 - c_var0 / (c_var0 + c_var1)
                w[c_items0] *= alpha
                w[c_items1] *= 1 - alpha
        return w

    @staticmethod
    def construct_hrp_portfolio(
        returns_df: pd.DataFrame,
        linkage_method: str = "ward",
        max_weight: float = 0.25,
        min_weight: float = 0.01,
    ) -> pd.Series:
        """
        Construct HRP portfolio using López de Prado's method.

        Steps:
        1. Compute covariance and correlation
        2. Hierarchical clustering on distance matrix
        3. Quasi-diagonal ordering
        4. Recursive bisection for weight allocation
        """
        cov = returns_df.cov()
        corr = returns_df.corr()
        dist = HRPStrategy.compute_distance_matrix(corr).values
        dist = squareform(dist, checks=False)
        link = linkage(dist, method=linkage_method)
        sort_ix = HRPStrategy.get_quasi_diag(link)
        sort_ix = [corr.index[i] for i in sort_ix]
        weights = HRPStrategy.get_rec_bipart(cov, sort_ix)
        weights = weights.clip(lower=min_weight, upper=max_weight)
        weights = weights / weights.sum()
        return weights


class RiskParityStrategy:
    """
    Risk Parity Strategy with Short Hedge.
    Equal risk contribution from each asset.
    Used in BEARISH (high volatility) regimes.
    """

    @staticmethod
    def compute_risk_contributions(weights: np.ndarray, cov: np.ndarray) -> np.ndarray:
        """Compute risk contribution of each asset to portfolio variance."""
        portfolio_vol = np.sqrt(np.dot(weights, np.dot(cov, weights)))
        marginal_contrib = np.dot(cov, weights)
        risk_contrib = weights * marginal_contrib / portfolio_vol
        return risk_contrib

    @staticmethod
    def optimize_risk_parity(
        cov: pd.DataFrame, max_iter: int = 1000, tolerance: float = 1e-06
    ) -> pd.Series:
        """
        Optimize for equal risk contribution using iterative algorithm.
        Assets with high risk get lower weight to equalize contributions.
        """
        n_assets = len(cov)
        weights = np.ones(n_assets) / n_assets
        cov_matrix = cov.values
        for iteration in range(max_iter):
            risk_contrib = RiskParityStrategy.compute_risk_contributions(weights, cov_matrix)
            target_risk = np.ones(n_assets) / n_assets
            adjustment = target_risk / (risk_contrib + 1e-10)
            new_weights = weights * adjustment
            new_weights = new_weights / new_weights.sum()
            if np.max(np.abs(new_weights - weights)) < tolerance:
                weights = new_weights
                break
            weights = new_weights
        return pd.Series(weights, index=cov.index)

    @staticmethod
    def construct_risk_parity_portfolio(
        returns_df: pd.DataFrame,
        long_exposure: float = 0.4,
        short_exposure: float = 0.15,
        short_ticker: str = "SH",
        max_weight: float = 0.25,
    ) -> pd.Series:
        """
        Construct risk parity portfolio with short hedge.

        Steps:
        1. Robust covariance estimation (Ledoit-Wolf)
        2. Optimize for equal risk contribution
        3. Scale to target long exposure
        4. Add short position in SH
        """
        lw = LedoitWolf()
        cov_matrix = lw.fit(returns_df.values).covariance_
        cov = pd.DataFrame(cov_matrix, index=returns_df.columns, columns=returns_df.columns)
        long_tickers = [t for t in returns_df.columns if t != short_ticker]
        cov_long = cov.loc[long_tickers, long_tickers]
        rp_weights = RiskParityStrategy.optimize_risk_parity(cov_long)
        rp_weights = rp_weights.clip(upper=max_weight)
        rp_weights = rp_weights / rp_weights.sum()
        rp_weights = rp_weights * long_exposure
        full_weights = pd.Series(0.0, index=returns_df.columns)
        full_weights[rp_weights.index] = rp_weights
        if short_ticker in returns_df.columns:
            full_weights[short_ticker] = short_exposure
        return full_weights


class RegimeAdaptiveBacktester:
    """
    Integrated backtester switching strategies based on detected regime.

    Regime 0 (BULLISH): OPS Momentum @ 120%
    Regime 1 (NEUTRAL): HRP @ 100%
    Regime 2 (BEARISH): Risk Parity + Short @ 55%
    """

    def __init__(self, config: Config):
        self.config = config

    def run_backtest(
        self, price_data: Dict[str, pd.DataFrame], regime_series: pd.Series, dates: pd.Series
    ) -> pd.DataFrame:
        """
        Run full regime-adaptive backtest.

        Parameters:
        -----------
        price_data : Dict[str, pd.DataFrame]
            Price data for each ticker
        regime_series : pd.Series
            Regime labels (0=Bullish, 1=Neutral, 2=Bearish)
        dates : pd.Series
            Date index

        Returns:
        --------
        pd.DataFrame
            Backtest results with portfolio values, weights, metrics
        """
        print("\n" + "=" * 80)
        print("REGIME-ADAPTIVE BACKTEST")
        print("=" * 80)
        aligned_prices = self._align_price_data(price_data, dates)
        aligned_regime = regime_series.loc[aligned_prices.index]
        print(
            f"Backtest Period: {aligned_prices.index[0].date()} to {aligned_prices.index[-1].date()}"
        )
        print(f"Trading Days: {len(aligned_prices)}")
        print(f"Trading Universe: {len(aligned_prices.columns)} assets")
        print(f"Rebalancing Frequency: Every {self.config.REBALANCE_FREQUENCY} days")
        print()
        initial_capital = 1000000
        portfolio_value = [initial_capital]
        portfolio_returns = [0.0]
        active_regime = [aligned_regime.iloc[0]]
        cash = [0.0]
        weight_history = {ticker: [0.0] for ticker in aligned_prices.columns}
        weight_history["cash"] = [1.0]
        n_rebalances = 0
        total_turnover = 0.0
        total_commissions = 0.0
        current_weights = pd.Series(0.0, index=aligned_prices.columns)
        lookback_window = 252
        for i in range(1, len(aligned_prices)):
            current_date = aligned_prices.index[i]
            current_regime = aligned_regime.iloc[i]
            is_rebalance_day = (
                i % self.config.REBALANCE_FREQUENCY == 0 or i == len(aligned_prices) - 1
            )
            price_changes = aligned_prices.iloc[i] / aligned_prices.iloc[i - 1] - 1
            portfolio_return = np.sum(current_weights * price_changes)
            new_portfolio_value = portfolio_value[-1] * (1 + portfolio_return)
            if is_rebalance_day and i >= lookback_window:
                historical_returns = (
                    aligned_prices.iloc[max(0, i - lookback_window) : i].pct_change().dropna()
                )
                if current_regime == 0:
                    new_weights = OPSMomentumStrategy.construct_momentum_portfolio(
                        historical_returns,
                        top_n=self.config.TOP_N_MOMENTUM,
                        min_score=self.config.MIN_MOMENTUM_SCORE,
                        max_weight=self.config.MAX_WEIGHT_SINGLE_ASSET,
                        target_vol=self.config.VOLATILITY_TARGET,
                        current_vol=historical_returns.std().mean() * np.sqrt(252),
                    )
                    new_weights = new_weights * self.config.BULLISH_EXPOSURE
                elif current_regime == 1:
                    new_weights = HRPStrategy.construct_hrp_portfolio(
                        historical_returns,
                        linkage_method=self.config.HRP_LINKAGE_METHOD,
                        max_weight=self.config.MAX_WEIGHT_SINGLE_ASSET,
                        min_weight=self.config.MIN_WEIGHT_SINGLE_ASSET,
                    )
                    new_weights = new_weights * self.config.NEUTRAL_EXPOSURE
                else:
                    new_weights = RiskParityStrategy.construct_risk_parity_portfolio(
                        historical_returns,
                        long_exposure=self.config.BEARISH_LONG_EXPOSURE,
                        short_exposure=self.config.BEARISH_SHORT_EXPOSURE,
                        short_ticker=self.config.SHORT_TICKER,
                        max_weight=self.config.MAX_WEIGHT_SINGLE_ASSET,
                    )
                turnover = np.abs(new_weights - current_weights).sum()
                total_turnover += turnover
                commission = turnover * new_portfolio_value * self.config.COMMISSION_PCT
                total_commissions += commission
                new_portfolio_value -= commission
                current_weights = new_weights
                n_rebalances += 1
            portfolio_value.append(new_portfolio_value)
            portfolio_returns.append(portfolio_return)
            active_regime.append(current_regime)
            for ticker in aligned_prices.columns:
                weight_history[ticker].append(current_weights.get(ticker, 0.0))
            weight_history["cash"].append(1.0 - current_weights.sum())
            cash.append((1.0 - current_weights.sum()) * new_portfolio_value)
        results_df = pd.DataFrame(
            {
                "date": aligned_prices.index,
                "portfolio_value": portfolio_value,
                "portfolio_return": portfolio_returns,
                "regime": active_regime,
                "cash": cash,
            }
        )
        for ticker, weights in weight_history.items():
            results_df[f"weight_{ticker}"] = weights
        self._print_performance_metrics(
            results_df, initial_capital, n_rebalances, total_turnover, total_commissions
        )
        return results_df

    def _align_price_data(
        self, price_data: Dict[str, pd.DataFrame], dates: pd.Series
    ) -> pd.DataFrame:
        """Align price data to common dates using inner join."""
        price_dfs = []
        for ticker, df in price_data.items():
            price_series = df.set_index("date")["adjClose"]
            price_series.name = ticker
            price_dfs.append(price_series)
        aligned = pd.concat(price_dfs, axis=1, join="inner")
        aligned = aligned.loc[aligned.index.isin(dates)]
        return aligned

    def _print_performance_metrics(
        self,
        results_df: pd.DataFrame,
        initial_capital: float,
        n_rebalances: int,
        total_turnover: float,
        total_commissions: float,
    ):
        """Print comprehensive performance metrics."""
        print("\n" + "=" * 80)
        print("BACKTEST PERFORMANCE METRICS")
        print("=" * 80)
        total_return = (results_df["portfolio_value"].iloc[-1] / initial_capital - 1) * 100
        n_years = len(results_df) / 252
        annual_return = (
            (results_df["portfolio_value"].iloc[-1] / initial_capital) ** (1 / n_years) - 1
        ) * 100
        print(f"\nTotal Return: {total_return:.2f}%")
        print(f"Annualized Return: {annual_return:.2f}%")
        returns_series = pd.Series(results_df["portfolio_return"].values[1:])
        sharpe_ratio = (
            returns_series.mean() / returns_series.std() * np.sqrt(252)
            if returns_series.std() > 0
            else 0
        )
        cumulative_returns = (1 + returns_series).cumprod()
        running_max = cumulative_returns.cummax()
        drawdown = (cumulative_returns - running_max) / running_max
        max_drawdown = drawdown.min() * 100
        daily_vol = returns_series.std() * np.sqrt(252) * 100
        print(f"Sharpe Ratio: {sharpe_ratio:.3f}")
        print(f"Annual Volatility: {daily_vol:.2f}%")
        print(f"Max Drawdown: {max_drawdown:.2f}%")
        print(f"\nNumber of Rebalances: {n_rebalances}")
        print(f"Total Turnover: {total_turnover:.2f}")
        print(
            f"Total Commissions: ${total_commissions:,.0f} ({100 * total_commissions / initial_capital:.3f}% of capital)"
        )
        print(f"Commission per Rebalance: ${total_commissions / n_rebalances:,.0f}")
        regime_counts = Counter(results_df["regime"])
        total_days = len(results_df)
        print(f"\nREGIME DISTRIBUTION:")
        print(
            f"  Bullish (Low Vol):  {regime_counts[0]:4d} days ({100 * regime_counts[0] / total_days:5.1f}%)"
        )
        print(
            f"  Neutral (Med Vol):  {regime_counts[1]:4d} days ({100 * regime_counts[1] / total_days:5.1f}%)"
        )
        print(
            f"  Bearish (High Vol): {regime_counts[2]:4d} days ({100 * regime_counts[2] / total_days:5.1f}%)"
        )
        print(f"\nPER-REGIME PERFORMANCE:")
        for regime_val in [0, 1, 2]:
            regime_mask = results_df["regime"] == regime_val
            regime_returns = results_df.loc[regime_mask, "portfolio_return"]
            if len(regime_returns) > 0:
                regime_name = ["Bullish", "Neutral", "Bearish"][regime_val]
                avg_return = regime_returns.mean() * 252 * 100
                regime_vol = regime_returns.std() * np.sqrt(252) * 100
                regime_sharpe = (
                    regime_returns.mean() / regime_returns.std() * np.sqrt(252)
                    if regime_returns.std() > 0
                    else 0
                )
                print(
                    f"  {regime_name:8s}: Ret={avg_return:+6.2f}%  Vol={regime_vol:5.2f}%  Sharpe={regime_sharpe:+5.2f}"
                )
        print("=" * 80)


class RegimeBacktestVisualizer:
    """Comprehensive visualization for regime-adaptive backtest."""

    @staticmethod
    def plot_spy_regime_detection(spy_data: pd.DataFrame, save_path: str = None):
        """
        Plot SPY regime detection results (4-panel chart).
        """
        fig, axes = plt.subplots(4, 1, figsize=(18, 12), sharex=True)
        regime_colors = {0: "lightgreen", 1: "yellow", 2: "red"}
        regime_names = {0: "Bullish (Low Vol)", 1: "Neutral (Med Vol)", 2: "Bearish (High Vol)"}
        ax1 = axes[0]
        ax1.plot(
            spy_data["date"],
            spy_data["adjClose"],
            linewidth=1.5,
            color="black",
            label="SPY Price",
            zorder=2,
        )
        for regime_val, color in regime_colors.items():
            mask = spy_data["regime"] == regime_val
            if mask.sum() > 0:
                ax1.fill_between(
                    spy_data["date"],
                    spy_data["adjClose"].min(),
                    spy_data["adjClose"].max(),
                    where=mask,
                    alpha=0.3,
                    color=color,
                    label=regime_names[regime_val],
                    zorder=1,
                )
        ax1.set_ylabel("SPY Price ($)", fontsize=11, fontweight="bold")
        ax1.legend(loc="upper left", fontsize=9, ncol=4)
        ax1.grid(True, alpha=0.3)
        ax1.set_title("SPY - BCPM Regime Detection Results", fontsize=14, fontweight="bold", pad=15)
        ax2 = axes[1]
        ax2.plot(
            spy_data["date"],
            spy_data["cp_prob_ensemble"],
            linewidth=2,
            color="darkblue",
            label="Ensemble CP Probability",
        )
        ax2.fill_between(spy_data["date"], 0, spy_data["cp_prob_ensemble"], alpha=0.3, color="blue")
        ax2.axhline(
            y=Config.CP_THRESHOLD_HIGH,
            color="red",
            linestyle="--",
            linewidth=1,
            alpha=0.7,
            label=f"High Threshold ({Config.CP_THRESHOLD_HIGH})",
        )
        ax2.set_ylabel("Changepoint\nProbability", fontsize=11, fontweight="bold")
        ax2.set_ylim([0, 1])
        ax2.legend(loc="upper right", fontsize=9)
        ax2.grid(True, alpha=0.3)
        ax3 = axes[2]
        if "rv_21_lag1" in spy_data.columns:
            ax3.plot(
                spy_data["date"],
                spy_data["rv_21_lag1"],
                linewidth=1.5,
                color="darkgreen",
                label="21-Day RV",
            )
            ax3.axhline(
                y=Config.LOW_VOL_THRESHOLD,
                color="green",
                linestyle="--",
                linewidth=1,
                alpha=0.7,
                label=f"Low Threshold ({Config.LOW_VOL_THRESHOLD})",
            )
            ax3.axhline(
                y=Config.HIGH_VOL_THRESHOLD,
                color="red",
                linestyle="--",
                linewidth=1,
                alpha=0.7,
                label=f"High Threshold ({Config.HIGH_VOL_THRESHOLD})",
            )
        ax3.set_ylabel("Realized\nVolatility", fontsize=11, fontweight="bold")
        ax3.legend(loc="upper left", fontsize=9)
        ax3.grid(True, alpha=0.3)
        ax4 = axes[3]
        if "run_length_medium" in spy_data.columns:
            ax4.plot(
                spy_data["date"],
                spy_data["run_length_medium"],
                linewidth=1.5,
                color="purple",
                label="Expected Run Length",
            )
        ax4.set_ylabel("Run Length\n(days)", fontsize=11, fontweight="bold")
        ax4.set_xlabel("Date", fontsize=11, fontweight="bold")
        ax4.legend(loc="upper left", fontsize=9)
        ax4.grid(True, alpha=0.3)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"  → Saved SPY regime plot to {save_path}")
        plt.show()

    @staticmethod
    def plot_backtest_results(
        results_df: pd.DataFrame, benchmark_df: pd.DataFrame = None, save_path: str = None
    ):
        """
        Plot comprehensive backtest results (4-panel chart).
        """
        fig, axes = plt.subplots(4, 1, figsize=(18, 14), sharex=True)
        regime_colors = {0: "lightgreen", 1: "yellow", 2: "red"}
        regime_names = {0: "Bullish", 1: "Neutral", 2: "Bearish"}
        ax1 = axes[0]
        portfolio_norm = 100 * results_df["portfolio_value"] / results_df["portfolio_value"].iloc[0]
        ax1.plot(
            results_df["date"],
            portfolio_norm,
            linewidth=2,
            color="darkblue",
            label="Regime-Adaptive Strategy",
            zorder=2,
        )
        if benchmark_df is not None:
            benchmark_norm = 100 * benchmark_df["adjClose"] / benchmark_df["adjClose"].iloc[0]
            ax1.plot(
                benchmark_df["date"],
                benchmark_norm,
                linewidth=1.5,
                color="gray",
                alpha=0.7,
                label="Buy & Hold (URTH)",
                zorder=1,
            )
        for regime_val, color in regime_colors.items():
            mask = results_df["regime"] == regime_val
            if mask.sum() > 0:
                ax1.fill_between(
                    results_df["date"],
                    portfolio_norm.min(),
                    portfolio_norm.max(),
                    where=mask,
                    alpha=0.2,
                    color=color,
                    label=regime_names[regime_val],
                    zorder=0,
                )
        ax1.set_ylabel("Portfolio Value\n(Normalized to 100)", fontsize=11, fontweight="bold")
        ax1.legend(loc="upper left", fontsize=9, ncol=3)
        ax1.grid(True, alpha=0.3)
        ax1.set_title(
            "Regime-Adaptive Portfolio Strategy - Full Backtest Results",
            fontsize=14,
            fontweight="bold",
            pad=15,
        )
        ax2 = axes[1]
        cumulative_returns = (1 + results_df["portfolio_return"]).cumprod()
        running_max = cumulative_returns.cummax()
        drawdown = (cumulative_returns - running_max) / running_max * 100
        ax2.fill_between(results_df["date"], 0, drawdown, alpha=0.3, color="red")
        ax2.plot(results_df["date"], drawdown, linewidth=1.5, color="darkred")
        ax2.set_ylabel("Drawdown (%)", fontsize=11, fontweight="bold")
        ax2.grid(True, alpha=0.3)
        ax3 = axes[2]
        regime_numeric = results_df["regime"].values
        ax3.fill_between(results_df["date"], 0, regime_numeric, alpha=0.5, step="post", linewidth=0)
        ax3.set_ylabel("Active Regime", fontsize=11, fontweight="bold")
        ax3.set_yticks([0, 1, 2])
        ax3.set_yticklabels(["Bullish", "Neutral", "Bearish"])
        ax3.grid(True, alpha=0.3, axis="x")
        ax4 = axes[3]
        weight_cols = [
            col for col in results_df.columns if col.startswith("weight_") and col != "weight_cash"
        ]
        if len(weight_cols) > 0:
            weight_data = results_df[weight_cols].values.T
            ticker_names = [col.replace("weight_", "") for col in weight_cols]
            ax4.stackplot(results_df["date"], weight_data, labels=ticker_names, alpha=0.8)
            ax4.set_ylabel("Asset Weights", fontsize=11, fontweight="bold")
            ax4.set_xlabel("Date", fontsize=11, fontweight="bold")
            ax4.legend(loc="upper left", fontsize=7, ncol=5, bbox_to_anchor=(0, 1))
            ax4.set_ylim([0, 1.5])
            ax4.grid(True, alpha=0.3)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"  → Saved backtest plot to {save_path}")
        plt.show()


def main():
    """Main execution function - complete regime-adaptive system."""
    print("=" * 80)
    print("REGIME-ADAPTIVE TRADING SYSTEM WITH BCPM V2.0")
    print("=" * 80)
    print()
    print("System Architecture:")
    print("  1. Detect regimes using BCPM on SPY ONLY")
    print("  2. Apply regime-specific strategies on expanded universe:")
    print("     • BULLISH (Low Vol <1%): OPS Momentum @ 120% exposure")
    print("     • NEUTRAL (Med Vol 1-2%): HRP @ 100% exposure")
    print("     • BEARISH (High Vol >2%): Risk Parity + SH Short @ 55% exposure")
    print()
    print("[STEP 1] Downloading data from Yahoo Finance...")
    print(f"Date Range: {Config.START_DATE} to {Config.END_DATE}")
    print()
    downloader = YahooDataDownloader()
    print("Downloading SPY for regime detection...")
    spy_data = downloader.download_ticker(Config.REGIME_TICKER, Config.START_DATE, Config.END_DATE)
    print("Downloading VIX...")
    vix_data = downloader.download_ticker(Config.VIX_TICKER, Config.START_DATE, Config.END_DATE)
    print()
    all_tickers = list(set(Config.TRADING_TICKERS + [Config.SHORT_TICKER]))
    trading_data = downloader.download_multiple_tickers(
        all_tickers, Config.START_DATE, Config.END_DATE
    )
    if spy_data.empty or len(trading_data) == 0:
        print("ERROR: Data download failed. Check API key and tickers.")
        return
    print()
    print("[STEP 2] Computing features for SPY regime detection...")
    feature_engine = EnhancedFeatureEngine()
    spy_data = feature_engine.compute_returns(spy_data)
    spy_data = feature_engine.compute_realized_volatility(spy_data, window=Config.RV_WINDOW)
    spy_data = feature_engine.compute_parkinson_volatility(spy_data, window=Config.PARKINSON_WINDOW)
    spy_data = feature_engine.compute_volume_ratio(spy_data, window=Config.VOLUME_MA_WINDOW)
    if not vix_data.empty:
        vix_data = feature_engine.compute_returns(vix_data)
        vix_data = feature_engine.compute_vix_features(vix_data, window=Config.VIX_SLOPE_WINDOW)
        spy_data = spy_data.merge(
            vix_data[["date", "adjClose"]], on="date", how="left", suffixes=("", "_vix")
        )
        spy_data = spy_data.rename(columns={"adjClose_vix": "VIX"})
    spy_data = feature_engine.lag_features(spy_data, lag=1)
    spy_data = spy_data.iloc[Config.MIN_HISTORY :].reset_index(drop=True)
    print(f"  → Prepared {len(spy_data)} days of SPY data")
    print()
    print("[STEP 3] Running BCPM regime detection on SPY...")
    print(f"  Multi-scale detection:")
    print(
        f"    Fast:   {Config.HAZARD_RATE_FAST:.5f} (expects change every ~{int(1 / Config.HAZARD_RATE_FAST)} days)"
    )
    print(
        f"    Medium: {Config.HAZARD_RATE_MEDIUM:.5f} (expects change every ~{int(1 / Config.HAZARD_RATE_MEDIUM)} days)"
    )
    print(
        f"    Slow:   {Config.HAZARD_RATE_SLOW:.5f} (expects change every ~{int(1 / Config.HAZARD_RATE_SLOW)} days)"
    )
    print(f"  Adaptive hazard: {Config.USE_ADAPTIVE_HAZARD}")
    print()
    detector = MultiScaleRegimeDetector()
    returns = spy_data["ret_close_lag1"].values
    vix_series = spy_data["VIX_lag1"].values if "VIX_lag1" in spy_data.columns else None
    volume_series = (
        spy_data["volume_ratio_lag1"].values if "volume_ratio_lag1" in spy_data.columns else None
    )
    bcpm_results = detector.run_multiscale_bocd(
        returns, vix_series=vix_series, volume_series=volume_series, config=Config
    )
    for key, values in bcpm_results.items():
        spy_data[key] = values
    regime_labels = detector.classify_regimes_3level(
        bcpm_results["regime_vol_ensemble"],
        low_threshold=Config.LOW_VOL_THRESHOLD,
        high_threshold=Config.HIGH_VOL_THRESHOLD,
        smoothing_window=Config.REGIME_SMOOTHING_WINDOW,
    )
    spy_data["regime"] = regime_labels
    print("Detecting changepoints for SPY...")
    n_cp_high = (bcpm_results["cp_prob_ensemble"] > Config.CP_THRESHOLD_HIGH).sum()
    n_cp_med = (bcpm_results["cp_prob_ensemble"] > Config.CP_THRESHOLD_MEDIUM).sum()
    regime_counts = Counter(regime_labels)
    print(f"  → High confidence changepoints: {n_cp_high}")
    print(f"  → Medium confidence changepoints: {n_cp_med}")
    print(
        f"  → Low vol: {regime_counts[0]} days ({100 * regime_counts[0] / len(regime_labels):.1f}%)"
    )
    print(
        f"  → Medium vol: {regime_counts[1]} days ({100 * regime_counts[1] / len(regime_labels):.1f}%)"
    )
    print(
        f"  → High vol: {regime_counts[2]} days ({100 * regime_counts[2] / len(regime_labels):.1f}%)"
    )
    print()
    print("[STEP 4] Preparing trading universe data...")
    for ticker in trading_data.keys():
        trading_data[ticker] = feature_engine.compute_returns(trading_data[ticker])
    print(f"  → Prepared {len(trading_data)} assets for trading")
    print()
    print("[STEP 5] Running regime-adaptive portfolio backtest...")
    regime_series = pd.Series(spy_data["regime"].values, index=spy_data["date"])
    backtester = RegimeAdaptiveBacktester(Config)
    results = backtester.run_backtest(trading_data, regime_series, spy_data["date"])
    if Config.SAVE_RESULTS:
        import os

        print("\n[STEP 6] Saving results...")
        os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
        results.to_csv(f"{Config.OUTPUT_DIR}backtest_results.csv", index=False)
        print(f"  → Saved backtest results")
        spy_data.to_csv(f"{Config.OUTPUT_DIR}spy_regime_detection.csv", index=False)
        print(f"  → Saved SPY regime detection data")
        print()
    print("[STEP 7] Creating visualizations...")
    print()
    visualizer = RegimeBacktestVisualizer()
    if Config.PLOT_SPY_REGIME:
        print("Plotting SPY regime detection...")
        save_path_spy = (
            f"{Config.OUTPUT_DIR}spy_regime_detection.png" if Config.SAVE_RESULTS else None
        )
        visualizer.plot_spy_regime_detection(spy_data, save_path=save_path_spy)
    print("Plotting backtest performance...")
    benchmark_data = downloader.download_ticker("URTH", Config.START_DATE, Config.END_DATE)
    save_path_backtest = (
        f"{Config.OUTPUT_DIR}backtest_performance.png" if Config.SAVE_RESULTS else None
    )
    visualizer.plot_backtest_results(
        results,
        benchmark_df=benchmark_data if not benchmark_data.empty else None,
        save_path=save_path_backtest,
    )
    print()
    print("=" * 80)
    print("REGIME-ADAPTIVE BACKTEST COMPLETE")
    print("=" * 80)
    print()
    print("Output Files:")
    print("  • backtest_results.csv - Full backtest time series")
    print("  • spy_regime_detection.csv - SPY regime detection data")
    print("  • spy_regime_detection.png - SPY regime visualization")
    print("  • backtest_performance.png - Portfolio performance visualization")
    print()
    print("Next Steps:")
    print("  • Review regime transitions and portfolio adjustments")
    print("  • Analyze per-regime performance metrics")
    print("  • Consider walk-forward optimization for robustness")
    print("  • Test sensitivity to parameter variations")
    print()


if __name__ == "__main__":
    main()
